# Mixed-Effects Models

A refresher on **mixed-effects models** (a.k.a. *multilevel*, *hierarchical*, or
*random-effects* models) — regression for data that comes in **groups**: repeated
measures per subject, students nested in schools, trials within items. Here we drive
them from Python with **statsmodels** `MixedLM`. For the R/`lme4` workflow see the
sibling notebook [[lme4]]; for the Bayesian version see [[pymc]].

**Domain:** Data Analysis & Research  ·  **runnable:** yes

## 1. What & Why

**What it is.** A mixed-effects model is a regression with two kinds of coefficients:

- **Fixed effects** — population-level parameters you want to estimate (the average
  effect of a drug, the slope of reaction time vs. sleep loss).
- **Random effects** — group-level deviations you treat as draws from a distribution
  (this *particular* subject runs 30 ms slow; that school scores 4 points high). You
  don't care about each level individually — you care about the *variance* across them.

**The problem it solves.** Ordinary least squares assumes every row is independent.
That breaks the moment your data has structure: 10 measurements from the same person
are more alike than 10 measurements from 10 people. Ignore it and your standard errors
are **too small** — you commit *pseudoreplication* and declare effects "significant"
that aren't. Mixed models put the clustering *into the model* so the uncertainty is
honest.

**When to reach for it.** Repeated measures, longitudinal/panel data, nested designs
(pupils in classes in schools), crossed designs (subjects × items in psycholinguistics),
or any time the same unit contributes multiple rows. **When not to:** one observation
per unit (nothing to pool), or a grouping factor with only 2–4 levels — you can't
estimate a variance from a handful of points, so model those as plain fixed effects.

## 2. Mental Model

**Fixed effects draw one line for everyone; random effects let each group wobble around
that line — and the size of the wobble is itself estimated and reined in.**

Imagine reaction-time-vs-day for 24 subjects. You have three choices:

- **Complete pooling** (`ols("rt ~ day")`): one line for the whole crowd. Throws away
  who's who, understates uncertainty.
- **No pooling** (`ols("rt ~ day * subject")`): a separate line per subject. Honors the
  groups but overfits — a subject with a fluke gets a wild slope, and you can't say
  anything about a *new* subject.
- **Partial pooling** (the mixed model): each subject gets their own line, but the
  estimates are **shrunk toward the population mean** by an amount the model learns from
  the data. Noisy/small groups shrink a lot; clean/large groups barely move.

Shrinkage is the whole point. The model estimates the between-group variance and the
within-group (residual) variance, and the ratio decides how much to trust each group's
own data versus the crowd. This is the same Bayesian-flavored idea behind hierarchical
models in [[pymc]] — just fit by (RE)ML here instead of sampling.

## 3. Key Concepts

- **Fixed vs random effects.** *Fixed* = the levels you want to estimate (drug vs.
  placebo). *Random* = a sample from a population you only want to account for (these 24
  subjects). Rule of thumb: would you want the *same* coefficient if you re-ran with new
  subjects? If yes → random.
- **Random intercept vs random slope.** A random **intercept** lets each group sit
  higher/lower (`groups="subject"`). A random **slope** lets the *effect* itself vary by
  group (`re_formula="~day"` → each subject has their own day-effect). Random slopes
  almost always also carry a random intercept and a correlation between the two.
- **Partial pooling & shrinkage / BLUPs.** Per-group estimates (Best Linear Unbiased
  Predictors) are conditional modes pulled toward 0. They're *predictions*, not free
  parameters — that's why you can estimate 24 of them from little data.
- **Variance components.** The model reports `Group Var` (between-subject intercept
  variance), slope variance, their covariance, and residual `Scale`. These — not the
  BLUPs — are the parameters.
- **ICC** (intraclass correlation) = between-group variance / total variance: the
  fraction of variation that lives *between* groups. High ICC ⇒ grouping matters a lot.
- **REML vs ML.** statsmodels fits **REML by default** (less biased variance estimates).
  Use ML (`reml=False`) when you need to compare models with **different fixed effects**
  via likelihood-ratio tests — REML likelihoods are only comparable across the same
  fixed-effects structure.
- **Crossed vs nested.** *Nested*: class within school (each class belongs to one
  school). *Crossed*: every subject sees every item. statsmodels handles one grouping
  factor cleanly; crossed/multiple factors need `vc_formula` and are where `lme4` is
  more comfortable.

## 4. Setup

`statsmodels` ships the `MixedLM` estimator (`statsmodels.formula.api.mixedlm`). It's
pure-Python + SciPy — no compiler, CPU-only, installs in seconds.

In [1]:
# %pip install statsmodels pandas numpy
import numpy as np
import pandas as pd
import statsmodels
import statsmodels.api as sm
import statsmodels.formula.api as smf

print("statsmodels", statsmodels.__version__)
print("pandas     ", pd.__version__)

statsmodels 0.14.6
pandas      2.3.3


## 5. Worked Examples

We simulate a classic repeated-measures design so we know the ground truth: 24 subjects
measured over 10 days. Reaction time rises with sleep deprivation, but **each subject has
their own baseline and their own rate of slowing**. That two-level structure is exactly
what a mixed model is built for.

### Example 1 — Random intercept

In [2]:
rng = np.random.default_rng(42)

n_subj, n_days = 24, 10
beta0, beta1 = 250.0, 10.0                 # true population intercept & slope (ms)
sd_intercept, sd_slope, sd_resid = 25.0, 5.0, 25.0

rows = []
for s in range(n_subj):
    u0 = rng.normal(0, sd_intercept)       # this subject's baseline offset
    u1 = rng.normal(0, sd_slope)           # this subject's slope offset
    for d in range(n_days):
        rt = (beta0 + u0) + (beta1 + u1) * d + rng.normal(0, sd_resid)
        rows.append((f"S{s:02d}", d, rt))

df = pd.DataFrame(rows, columns=["subject", "day", "rt"])
print(df.head())
print(f"\n{n_subj} subjects x {n_days} days = {len(df)} rows")

# Random INTERCEPT model: each subject gets their own baseline; one shared slope.
m_ri = smf.mixedlm("rt ~ day", df, groups="subject").fit()
print(m_ri.summary())

  subject  day          rt
0     S00    0  276.379207
1     S00    1  285.932124
2     S00    2  218.442206
3     S00    3  239.463678
4     S00    4  280.014255

24 subjects x 10 days = 240 rows
          Mixed Linear Model Regression Results
Model:              MixedLM Dependent Variable: rt        
No. Observations:   240     Method:             REML      
No. Groups:         24      Scale:              802.4798  
Min. group size:    10      Log-Likelihood:     -1170.2960
Max. group size:    10      Converged:          Yes       
Mean group size:    10.0                                  
----------------------------------------------------------
             Coef.   Std.Err.   z    P>|z|  [0.025  0.975]
----------------------------------------------------------
Intercept    241.399    7.381 32.706 0.000 226.933 255.866
day           10.265    0.637 16.123 0.000   9.017  11.512
subject Var 1030.272   12.162                             



Read the summary like this: the `day` coefficient (~10) is the fixed slope — the average
ms-per-day across the population. `Group Var` is the estimated **between-subject intercept
variance**; its square root should land near our true `sd_intercept = 25`. `Scale` is the
residual variance. The standard error on `day` is now computed *acknowledging* that the 10
rows per subject aren't independent — that's the honesty pooled OLS lacks.

### Example 2 — Random slopes, shrinkage, and the pooled-OLS comparison

In [3]:
# Random intercept AND random slope for `day`, per subject.
m_rs = smf.mixedlm("rt ~ day", df, groups="subject", re_formula="~day").fit()
print(m_rs.summary())

# Complete pooling ignores grouping entirely -> over-confident SE on the slope.
m_ols = smf.ols("rt ~ day", df).fit()
print("\nSlope estimate (ground truth = 10.0):")
print("  Pooled OLS : %.2f  (SE %.2f)" % (m_ols.params["day"], m_ols.bse["day"]))
print("  Mixed (RS) : %.2f  (SE %.2f)" % (m_rs.fe_params["day"], m_rs.bse_fe["day"]))

             Mixed Linear Model Regression Results
Model:               MixedLM   Dependent Variable:   rt        
No. Observations:    240       Method:               REML      
No. Groups:          24        Scale:                544.7251  
Min. group size:     10        Log-Likelihood:       -1141.5719
Max. group size:     10        Converged:            Yes       
Mean group size:     10.0                                      
---------------------------------------------------------------
                   Coef.  Std.Err.   z    P>|z|  [0.025  0.975]
---------------------------------------------------------------
Intercept         241.399    5.224 46.206 0.000 231.159 251.639
day                10.265    1.221  8.404 0.000   7.871  12.659
subject Var       466.896    8.758                             
subject x day Cov  -0.243    1.400                             
day Var            29.205    0.479                             


Slope estimate (ground truth = 10.0):
  Pooled OLS 

The point estimates agree, but the **mixed model's SE on the slope is larger** — because
it knows the slope was measured on only 24 independent subjects, not 240 independent rows.
That bigger-but-correct SE is the difference between a real finding and pseudoreplication.

Now the headline behavior: **shrinkage**. Fit each subject in isolation (no pooling) and
compare their slopes to the partially-pooled mixed estimates.

In [4]:
# Per-subject random-effect deviations (BLUPs): conditional modes, shrunk toward 0.
re = pd.DataFrame(m_rs.random_effects).T          # columns: 'Group' (intercept), 'day'

# No pooling: an independent OLS slope for each subject.
no_pool = pd.Series({s: np.polyfit(g.day, g.rt, 1)[0] for s, g in df.groupby("subject")})

# Partial pooling: population slope + each subject's BLUP slope deviation.
partial = m_rs.fe_params["day"] + re["day"]

comp = pd.DataFrame({"no_pooling": no_pool, "partial_pooling": partial}).round(2)
print(comp.head(8))
print("\nSpread of per-subject slopes (true between-subject sd = 5.0):")
print("  no-pooling      sd : %.2f" % comp.no_pooling.std())
print("  partial-pooling sd : %.2f   <- shrunk toward the population mean 10.0"
      % comp.partial_pooling.std())

     no_pooling  partial_pooling
S00        6.12             7.05
S01       16.41            15.74
S02        7.57             8.39
S03        7.01             7.72
S04       11.91            12.31
S05        3.91             4.33
S06       10.47            10.23
S07        8.36             8.29

Spread of per-subject slopes (true between-subject sd = 5.0):
  no-pooling      sd : 5.99
  partial-pooling sd : 4.98   <- shrunk toward the population mean 10.0


The partial-pooling slopes are **less spread out** than the no-pooling ones: subjects
whose raw slope was extreme (driven by within-subject noise) get pulled back toward the
group mean. That regularization is why mixed-model per-group estimates predict *new* data
better than fitting each group alone.

### Optional — fit the canonical `sleepstudy` data (gated: small download)

In [5]:
import os

# The real Belenky et al. sleep-deprivation study is bundled with lme4. Fetching it
# hits the network, so we gate it behind an env flag to keep this notebook offline-safe.
if os.getenv("FETCH_RDATA"):
    data = sm.datasets.get_rdataset("sleepstudy", "lme4").data
    m = smf.mixedlm("Reaction ~ Days", data, groups="Subject", re_formula="~Days").fit()
    print(m.summary())
else:
    print("Set FETCH_RDATA=1 to download lme4::sleepstudy and refit on real data.")
    print("Call shape: smf.mixedlm('Reaction ~ Days', data, groups='Subject',")
    print("                        re_formula='~Days').fit()")

Set FETCH_RDATA=1 to download lme4::sleepstudy and refit on real data.
Call shape: smf.mixedlm('Reaction ~ Days', data, groups='Subject',
                        re_formula='~Days').fit()


## 6. Gotchas & Pitfalls

- **Singular / boundary fits.** When a variance is estimated at ~0 (or a correlation at
  ±1), the model is **overparameterized** for your data — too few groups or too little
  signal to support that random term. Simplify (drop a random slope) rather than ignore
  the warning.
- **statsmodels is not `lme4`.** `MixedLM` handles **one** grouping factor natively;
  crossed random effects need `vc_formula` gymnastics, and there's **no general GLMM**
  (Poisson/logistic mixed models). For those, reach for R's [[lme4]] or Bayesian [[pymc]].
- **REML likelihoods aren't comparable across different fixed effects.** To compare models
  that differ in fixed effects, refit with `reml=False` (ML) before any likelihood-ratio
  test or AIC comparison.
- **Don't randomize a factor with too few levels.** With <5–6 groups you can't estimate a
  variance; treat that factor as a fixed effect instead.
- **BLUPs aren't parameters.** `random_effects` are conditional modes (predictions) — they
  have no naive standard errors and shouldn't be reported as if they were fitted
  coefficients.
- **Center your predictors.** A random intercept's variance is the spread of group lines
  *at predictor = 0*. If `day` ranged 1990–2020, "the intercept" is nonsense; center first
  so the intercept means something.
- **p-values are contested.** The denominator degrees of freedom for fixed effects in
  mixed models are genuinely hard; statsmodels uses a normal approximation. For small
  samples, treat marginal p-values with care (this is why R's `lmerTest` exists).

## 7. When to Use vs Alternatives

| Approach | Use when | Trade-off |
|---|---|---|
| **Pooled OLS** | No grouping, or you genuinely don't care about clusters | Wrong (too small) SEs under clustering → pseudoreplication |
| **Fixed-effects dummies** (one per group) | Few groups you want to estimate individually; econometric "within" estimator | Costs a parameter per group, can't generalize to new groups, no shrinkage |
| **Mixed-effects model** (this notebook) | Many groups, repeated/nested/longitudinal data, want population effects *and* group-level predictions | One grouping factor is easy in statsmodels; GLMM/crossed designs are awkward |
| **GEE** (`statsmodels.GEE`) | You only want **population-average** effects + robust SEs, not group-level estimates | Gives marginal (not subject-specific) coefficients; no BLUPs/variance components |
| **Bayesian hierarchical** ([[pymc]]) | Small data, complex nesting, want full posterior uncertainty on variance components | Slower (sampling), needs priors and convergence checks |
| **R `lme4`** ([[lme4]]) | Crossed random effects, GLMMs, the mature ecosystem (`lmerTest`, `emmeans`) | Means leaving Python |

**Rule of thumb:** clustered data + you care about both the average effect and the spread
across groups → mixed model. Only the average effect with messy correlation → GEE. Need
GLMM or crossed effects in Python → consider PyMC; otherwise R's `lme4` is the gold
standard.

## 8. Resources

- **statsmodels — Linear Mixed Effects Models** (API + theory):
  https://www.statsmodels.org/stable/mixed_linear.html
- **statsmodels `MixedLM` worked example notebook**:
  https://www.statsmodels.org/stable/examples/notebooks/generated/mixed_lm_example.html
- **Bates, Mächler, Bolker & Walker — "Fitting Linear Mixed-Effects Models Using lme4"**
  (JSS, the canonical reference): https://www.jstatsoft.org/article/view/v067i01
- **Ben Bolker's GLMM FAQ** (the field's collective wisdom on pitfalls):
  https://bbolker.github.io/mixedmodels-misc/glmmFAQ.html
- **Gelman & Hill — *Data Analysis Using Regression and Multilevel/Hierarchical Models***
  (the textbook on partial pooling): http://www.stat.columbia.edu/~gelman/arm/